In [23]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal


# =========================================================
# Config
# =========================================================
ROOT_DIR = Path(r"D:\mmwave-heart-rate-monitoring-demo")
DATA_DIR = ROOT_DIR / "data"
OUT_DIR = ROOT_DIR / "results" / "alignment"

COND_NAMES = [
    "AWR_steady",
    "AWR_unsteady",
    "IWR_steady",
    "IWR_unsteady",
]

HR_MIN_BPM = 40.0
HR_MAX_BPM = 180.0
ZERO_PAD_FACTOR = 8

# alignment
FS_COMMON = 100.0
DUR_ECG = 30.0
DUR_MMW = 60.0

# DTW
DTW_DS_FS = 20.0
DTW_COST_CAP = 9.0

# DR-SCC
WIN_SEC = 10.0
HOP_SEC = 1.0
MAX_STEP_SEC = 0.10
LAM = 0.15

# XCORR / GCC-PHAT / Peaks+Drift
MAXLAG_S = 5.0
PEAKS_DRIFT_WIN_S = 10.0
PEAKS_DRIFT_STEP_S = 5.0

# plot
COLOR_MMW = "#1f3b73"
COLOR_ECG = "#6b7280"
SHADE = "#c7dbf5"

In [24]:
# =========================================================
# Helpers
# =========================================================
def sample_key_from_name(name: str) -> int:
    m = re.search(r"sample_(\d+)", name)
    return int(m.group(1)) if m else 10**9


def sample_key(p: Path) -> int:
    return sample_key_from_name(p.name)


def format_seg(start: float, end: float) -> str:
    if not (np.isfinite(start) and np.isfinite(end)):
        return ""
    return f"[{start:.2f}, {end:.2f}]"


def blank_col(n: int) -> str:
    return " " * n


def estimate_fs(t: np.ndarray) -> float:
    dt = np.diff(t)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) == 0:
        raise ValueError("time 欄位無法估計取樣率")
    return 1.0 / np.median(dt)


def normalize_pm1(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    x = x - np.nanmedian(x)
    mx = np.nanmax(np.abs(x)) + 1e-12
    return x / mx


def resample_to(t_src: np.ndarray, x_src: np.ndarray, t_dst: np.ndarray) -> np.ndarray:
    return np.interp(t_dst, t_src, x_src)


def _zscore(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    m = np.nanmean(x)
    s = np.nanstd(x) + 1e-12
    return (x - m) / s

In [25]:
# =========================================================
# 檔案讀取
# =========================================================
def find_waveform_csv(sample_dir: Path) -> Optional[Path]:
    cand = sample_dir / "waveform.csv"
    if cand.exists():
        return cand

    hits = list(sample_dir.glob("*waveform*.csv"))
    if hits:
        return hits[0]

    return None


def find_ecg_csv(cond_dir: Path, sample_name: str) -> Optional[Path]:
    cand = cond_dir / "raw" / "ECG" / f"{sample_name}.csv"
    if cand.exists():
        return cand

    hits = list((cond_dir / "raw" / "ECG").glob(f"{sample_name}*.csv"))
    if hits:
        return hits[0]

    hits2 = list(cond_dir.glob(f"**/{sample_name}.csv"))
    hits2 = [p for p in hits2 if "raw" in str(p).lower() and "ecg" in str(p).lower()]
    if hits2:
        return hits2[0]

    return None


def load_waveform(csv_path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path, comment="#")

    cols_lower = {c.lower(): c for c in df.columns}
    if "time" not in cols_lower:
        raise ValueError(f"{csv_path} 缺少 time 欄位")
    if "lsb" not in cols_lower:
        raise ValueError(f"{csv_path} 缺少 LSB 欄位")

    time_col = cols_lower["time"]
    sig_col = cols_lower["lsb"]

    t = pd.to_numeric(df[time_col], errors="coerce").to_numpy(dtype=float)
    x = pd.to_numeric(df[sig_col], errors="coerce").to_numpy(dtype=float)

    mask = np.isfinite(t) & np.isfinite(x)
    t = t[mask]
    x = x[mask]

    if len(t) < 8:
        raise ValueError(f"{csv_path} 有效資料太少")

    t = t - t[0]
    return t, x


def read_ecg_csv(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)

    t = pd.to_numeric(df.iloc[:, 0], errors="coerce").to_numpy(float)
    x = pd.to_numeric(df.iloc[:, 1], errors="coerce").to_numpy(float)

    m = np.isfinite(t) & np.isfinite(x)
    t, x = t[m], x[m]

    if len(t) < 8:
        raise ValueError(f"{path} ECG 有效資料太少")

    t = t - t[0]
    return t, x

In [26]:
# =========================================================
# FFT HR 方法（只用在對齊後 mmWave 30s 區段）
# =========================================================
def compute_fft_spectrum(
    t: np.ndarray,
    x: np.ndarray,
    zero_pad_factor: int = ZERO_PAD_FACTOR,
) -> Tuple[float, np.ndarray, np.ndarray]:
    fs = estimate_fs(t)

    n = len(x)
    n_fft = int(2 ** np.ceil(np.log2(n * zero_pad_factor)))

    spec = np.fft.rfft(x, n=n_fft)
    freqs_hz = np.fft.rfftfreq(n_fft, d=1.0 / fs)
    power = np.abs(spec) ** 2

    return fs, freqs_hz, power


def estimate_hr_from_band(
    freqs_hz: np.ndarray,
    power: np.ndarray,
    hr_min_bpm: float = HR_MIN_BPM,
    hr_max_bpm: float = HR_MAX_BPM,
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    fmin = hr_min_bpm / 60.0
    fmax = hr_max_bpm / 60.0

    band_mask = (freqs_hz >= fmin) & (freqs_hz <= fmax)
    freqs_band = freqs_hz[band_mask]
    power_band = power[band_mask]

    if len(freqs_band) == 0:
        raise ValueError("HR 搜尋頻帶內沒有頻率點")

    peak_idx = int(np.argmax(power_band))
    peak_freq_hz = float(freqs_band[peak_idx])
    hr_bpm = peak_freq_hz * 60.0

    return freqs_band, power_band, peak_freq_hz, hr_bpm


def estimate_hr_fft_from_signal(
    t: np.ndarray,
    x: np.ndarray,
    hr_min_bpm: float = HR_MIN_BPM,
    hr_max_bpm: float = HR_MAX_BPM,
    zero_pad_factor: int = ZERO_PAD_FACTOR,
) -> float:
    _, freqs_hz, power = compute_fft_spectrum(
        t, x, zero_pad_factor=zero_pad_factor
    )
    _, _, _, hr_bpm = estimate_hr_from_band(
        freqs_hz, power,
        hr_min_bpm=hr_min_bpm,
        hr_max_bpm=hr_max_bpm
    )
    return float(hr_bpm)

In [27]:
# =========================================================
# 五種對齊方法
# =========================================================
def align_global_xcorr_fullsearch(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float):
    
    N_m = len(mmw_sig_60)
    N_e = len(ecg_sig_30)

    if N_m < N_e:
        return np.nan

    best_score = -np.inf
    best_shift = 0

    # brute-force sliding correlation
    for s in range(N_m - N_e + 1):

        score = 0.0

        for i in range(N_e):
            score += mmw_sig_60[s + i] * ecg_sig_30[i]

        if score > best_score:
            best_score = score
            best_shift = s

    start_time = best_shift / fs


    return float(start_time)


def align_dr_scc(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float):
    N_m = mmw_sig_60.size
    N_e = ecg_sig_30.size
    if N_m <= N_e + 2:
        return None, None, None

    tau0 = align_global_xcorr_fullsearch(mmw_sig_60, ecg_sig_30, fs)
    
    if not np.isfinite(tau0):
        tau0 = 0.0
    s_prev = int(round(tau0 * fs))

    win = int(round(WIN_SEC * fs))
    hop = int(round(HOP_SEC * fs))
    max_step = int(round(MAX_STEP_SEC * fs))

    starts = np.arange(0, max(1, N_e - win + 1), hop)

    x_all = _zscore(ecg_sig_30)
    y_all = _zscore(mmw_sig_60)

    shift_samples = []
    shift_centers = []

    s_min = 0
    s_max = N_m - N_e
    s_prev = int(np.clip(s_prev, s_min, s_max))

    for st in starts:
        ed = st + win
        xw = x_all[st:ed]

        cand_lo = max(s_min, s_prev - max_step)
        cand_hi = min(s_max, s_prev + max_step)
        cands = np.arange(cand_lo, cand_hi + 1)

        best_score = -1e18
        best_s = s_prev

        for s in cands:
            yw = y_all[s + st : s + ed]
            if yw.size != xw.size:
                continue

            corr = float(np.dot(xw, yw) / (xw.size + 1e-12))
            score = corr - LAM * (abs(s - s_prev) / (max_step + 1e-12))

            if score > best_score:
                best_score = score
                best_s = int(s)

        shift_samples.append(best_s)
        shift_centers.append(int(st + win // 2))
        s_prev = best_s

    shift_centers = np.array(shift_centers, dtype=int)
    shift_samples = np.array(shift_samples, dtype=float)

    idx = np.arange(N_e)
    shift_per_sample = np.interp(
        idx, shift_centers, shift_samples,
        left=shift_samples[0], right=shift_samples[-1]
    )
    tau_per_sample = shift_per_sample / fs
    return tau_per_sample, shift_centers / fs, shift_samples / fs


def align_dtw_subseq_bounds(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float, ds_fs: float = DTW_DS_FS):
    if mmw_sig_60.size < ecg_sig_30.size + 10:
        return np.nan, np.nan

    ds = max(1, int(round(fs / ds_fs)))
    y = mmw_sig_60[::ds].astype(np.float32)
    x = ecg_sig_30[::ds].astype(np.float32)

    if x.size < 40 or y.size < x.size + 10:
        return np.nan, np.nan

    y = _zscore(y)
    x = _zscore(x)

    N = int(x.size)
    M = int(y.size)

    D = np.full((N + 1, M + 1), np.inf, dtype=np.float32)
    P = np.zeros((N + 1, M + 1), dtype=np.uint8)
    D[0, :] = 0.0

    for i in range(1, N + 1):
        xi = float(x[i - 1])
        for j in range(1, M + 1):
            d = xi - float(y[j - 1])
            c = d * d
            if c > DTW_COST_CAP:
                c = DTW_COST_CAP

            a = D[i - 1, j - 1]
            b = D[i - 1, j]
            l = D[i, j - 1]

            if a <= b and a <= l:
                D[i, j] = c + a
                P[i, j] = 0
            elif b <= l:
                D[i, j] = c + b
                P[i, j] = 1
            else:
                D[i, j] = c + l
                P[i, j] = 2

    end_j = 1 + int(np.argmin(D[N, 1:]))

    i, j = N, end_j
    while i > 0 and j > 0:
        move = P[i, j]
        if move == 0:
            i -= 1
            j -= 1
        elif move == 1:
            i -= 1
        else:
            j -= 1

    start_j = max(1, j)
    start_sec = (start_j - 1) * ds / fs
    end_sec = (end_j - 1) * ds / fs

    if not (np.isfinite(start_sec) and np.isfinite(end_sec)) or end_sec <= start_sec:
        return np.nan, np.nan

    return float(start_sec), float(end_sec)


def align_xcorr_start(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float) -> float:
    return align_global_xcorr_fullsearch(mmw_sig_60, ecg_sig_30, fs)


def align_gccphat_start(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float) -> float:
    N_m = len(mmw_sig_60)
    N_e = len(ecg_sig_30)
    if N_m < N_e:
        return np.nan

    x = _zscore(ecg_sig_30)
    best_score = -np.inf
    best_start = np.nan

    for s in range(0, N_m - N_e + 1):
        y = _zscore(mmw_sig_60[s:s + N_e])

        n = int(2 ** np.ceil(np.log2(len(x) + len(y))))
        X = np.fft.rfft(np.nan_to_num(x), n)
        Y = np.fft.rfft(np.nan_to_num(y), n)
        R = X * np.conj(Y)
        R /= np.abs(R) + 1e-12
        cc = np.fft.irfft(R, n)

        score = float(np.nanmax(np.abs(cc)))
        if score > best_score:
            best_score = score
            best_start = s / fs

    return float(best_start) if np.isfinite(best_start) else np.nan


def align_peaks_drift_start(mmw_sig_60: np.ndarray, ecg_sig_30: np.ndarray, fs: float) -> float:
    start0 = align_global_xcorr_fullsearch(mmw_sig_60, ecg_sig_30, fs)
    if not np.isfinite(start0):
        return np.nan

    N_e = len(ecg_sig_30)
    s0 = int(round(start0 * fs))
    if s0 + N_e > len(mmw_sig_60):
        s0 = len(mmw_sig_60) - N_e
    if s0 < 0:
        return np.nan

    mmw_seg = mmw_sig_60[s0:s0 + N_e]
    ecg_seg = ecg_sig_30

    t_ecg = np.arange(len(ecg_seg)) / fs
    win = max(1, int(round(PEAKS_DRIFT_WIN_S * fs)))
    step = max(1, int(round(PEAKS_DRIFT_STEP_S * fs)))

    centers = []
    taus = []

    for st in range(0, len(ecg_seg) - win + 1, step):
        ed = st + win
        xw = _zscore(ecg_seg[st:ed])
        yw = _zscore(mmw_seg[st:ed])

        c = signal.correlate(xw, yw, mode="full")
        l = signal.correlation_lags(len(xw), len(yw), mode="full") / fs
        m = np.abs(l) <= MAXLAG_S

        if np.any(m) and np.isfinite(c[m]).any():
            tau = float(l[m][np.nanargmax(c[m])])
        else:
            tau = 0.0

        centers.append((st + ed) / 2 / fs)
        taus.append(tau)

    centers = np.asarray(centers, float)
    taus = np.asarray(taus, float)

    if len(centers) < 2:
        return float(start0)

    A = np.vstack([centers, np.ones_like(centers)]).T
    a, b = np.linalg.lstsq(A, taus, rcond=None)[0]

    t_aligned = t_ecg - (a * t_ecg + b)
    start_corr = start0 + np.nanmin(t_aligned)
    start_corr = float(np.clip(start_corr, 0.0, DUR_MMW - DUR_ECG))
    return start_corr

In [28]:
# =========================================================
# 單一 sample 分析
# =========================================================
def compute_segment_hr_from_raw_mmw(
    t_mmw: np.ndarray,
    x_mmw: np.ndarray,
    start_s: float,
    dur_s: float = DUR_ECG,
) -> float:
    if not np.isfinite(start_s):
        return np.nan

    end_s = start_s + dur_s
    m = (t_mmw >= start_s) & (t_mmw < end_s)
    if np.sum(m) < 8:
        return np.nan

    try:
        return estimate_hr_fft_from_signal(t_mmw[m], x_mmw[m])
    except Exception:
        return np.nan


def prepare_signals_for_alignment(
    mmw_csv: Path,
    ecg_csv: Path,
) -> Dict[str, Any]:
    t_mmw, x_mmw = load_waveform(mmw_csv)
    t_ecg, x_ecg = read_ecg_csv(ecg_csv)

    # ECG 只取前 30 秒
    t_ecg_30 = t_ecg[t_ecg <= DUR_ECG]
    x_ecg_30 = x_ecg[:t_ecg_30.size]

    # 共同時間軸
    t_mmw_c = np.arange(0, DUR_MMW, 1.0 / FS_COMMON)
    mmw_c = resample_to(t_mmw, x_mmw, t_mmw_c)

    ecg_end = min(DUR_ECG, t_ecg_30[-1] if len(t_ecg_30) else DUR_ECG)
    t_ecg_c = np.arange(0, ecg_end, 1.0 / FS_COMMON)
    ecg_c = resample_to(t_ecg_30, x_ecg_30, t_ecg_c)

    return {
        "t_mmw": t_mmw,
        "x_mmw": x_mmw,
        "t_ecg": t_ecg_30,
        "x_ecg": x_ecg_30,
        "t_mmw_c": t_mmw_c,
        "mmw_c": mmw_c,
        "t_ecg_c": t_ecg_c,
        "ecg_c": ecg_c,
        "mmw_norm_axis": normalize_pm1(mmw_c),
        "ecg_norm_30": normalize_pm1(ecg_c),
    }


def analyze_one_sample(
    mmw_csv: Path,
    ecg_csv: Path,
) -> Dict[str, Any]:
    sig = prepare_signals_for_alignment(mmw_csv, ecg_csv)

    t_mmw = sig["t_mmw"]
    x_mmw = sig["x_mmw"]
    mmw_c = sig["mmw_c"]
    ecg_c = sig["ecg_c"]

    # DTW
    start_dtw, end_dtw = align_dtw_subseq_bounds(mmw_c, ecg_c, FS_COMMON, ds_fs=DTW_DS_FS)
    if np.isfinite(start_dtw):
        start_dtw = float(np.clip(start_dtw, 0.0, DUR_MMW - DUR_ECG))
        end_dtw = start_dtw + DUR_ECG
    else:
        end_dtw = np.nan
    hr_dtw = compute_segment_hr_from_raw_mmw(t_mmw, x_mmw, start_dtw)

    # DR-SCC
    tau_per_sample, _, _ = align_dr_scc(mmw_c, ecg_c, FS_COMMON)
    if tau_per_sample is None:
        start_drscc = np.nan
        end_drscc = np.nan
        t_ecg_drscc = sig["t_ecg_c"] + np.nan
    else:
        t_ecg_drscc = sig["t_ecg_c"] + tau_per_sample
        start_drscc = float(np.nanmin(t_ecg_drscc))
        start_drscc = float(np.clip(start_drscc, 0.0, DUR_MMW - DUR_ECG))
        end_drscc = start_drscc + DUR_ECG
    hr_drscc = compute_segment_hr_from_raw_mmw(t_mmw, x_mmw, start_drscc)

    # XCORR
    start_xcorr = align_xcorr_start(mmw_c, ecg_c, FS_COMMON)
    if np.isfinite(start_xcorr):
        start_xcorr = float(np.clip(start_xcorr, 0.0, DUR_MMW - DUR_ECG))
        end_xcorr = start_xcorr + DUR_ECG
    else:
        end_xcorr = np.nan
    hr_xcorr = compute_segment_hr_from_raw_mmw(t_mmw, x_mmw, start_xcorr)

    # GCC-PHAT
    start_gcc = align_gccphat_start(mmw_c, ecg_c, FS_COMMON)
    if np.isfinite(start_gcc):
        start_gcc = float(np.clip(start_gcc, 0.0, DUR_MMW - DUR_ECG))
        end_gcc = start_gcc + DUR_ECG
    else:
        end_gcc = np.nan
    hr_gcc = compute_segment_hr_from_raw_mmw(t_mmw, x_mmw, start_gcc)

    # Peaks + Drift
    start_peaks = align_peaks_drift_start(mmw_c, ecg_c, FS_COMMON)
    if np.isfinite(start_peaks):
        start_peaks = float(np.clip(start_peaks, 0.0, DUR_MMW - DUR_ECG))
        end_peaks = start_peaks + DUR_ECG
    else:
        end_peaks = np.nan
    hr_peaks = compute_segment_hr_from_raw_mmw(t_mmw, x_mmw, start_peaks)

    return {
        **sig,

        "start_dtw": start_dtw,
        "end_dtw": end_dtw,
        "mmw_HR_dtw": hr_dtw,

        "start_drscc": start_drscc,
        "end_drscc": end_drscc,
        "mmw_HR_drscc": hr_drscc,
        "t_ecg_drscc": t_ecg_drscc,

        "start_xcorr": start_xcorr,
        "end_xcorr": end_xcorr,
        "mmw_HR_xcorr": hr_xcorr,

        "start_gccphat": start_gcc,
        "end_gccphat": end_gcc,
        "mmw_HR_gccphat": hr_gcc,

        "start_peaks_drift": start_peaks,
        "end_peaks_drift": end_peaks,
        "mmw_HR_peaks_drift": hr_peaks,
    }

In [29]:
# =========================================================
# 視覺化
# =========================================================
def plot_one_alignment_subplot(
    ax,
    t_mmw_axis: np.ndarray,
    mmw_norm_axis: np.ndarray,
    t_ecg_aligned: np.ndarray,
    ecg_norm_30: np.ndarray,
    title: str,
    shade_start: float,
    shade_end: float,
):
    if np.isfinite(shade_start) and np.isfinite(shade_end):
        ax.axvspan(shade_start, shade_end, color=SHADE, alpha=0.55, zorder=0)

    ax.plot(t_ecg_aligned, ecg_norm_30, color=COLOR_ECG, linewidth=1.0, alpha=0.55, label="ECG", zorder=1)
    ax.plot(t_mmw_axis, mmw_norm_axis, color=COLOR_MMW, linewidth=1.4, alpha=0.95, label="mmWave", zorder=2)

    ax.set_xlim(0, DUR_MMW)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Norm amp")
    ax.grid(True, linewidth=0.4, alpha=0.5)


def plot_all_5_methods(
    result: Dict[str, Any],
    sample_name: str,
    cond_name: str,
    save_path: Optional[Path] = None,
):
    t_mmw_c = result["t_mmw_c"]
    mmw_norm_axis = result["mmw_norm_axis"]
    t_ecg_c = result["t_ecg_c"]
    ecg_norm_30 = result["ecg_norm_30"]

    t_ecg_dtw = t_ecg_c + result["start_dtw"] if np.isfinite(result["start_dtw"]) else t_ecg_c + np.nan
    t_ecg_xcorr = t_ecg_c + result["start_xcorr"] if np.isfinite(result["start_xcorr"]) else t_ecg_c + np.nan
    t_ecg_gcc = t_ecg_c + result["start_gccphat"] if np.isfinite(result["start_gccphat"]) else t_ecg_c + np.nan
    t_ecg_peaks = t_ecg_c + result["start_peaks_drift"] if np.isfinite(result["start_peaks_drift"]) else t_ecg_c + np.nan
    t_ecg_drscc = result["t_ecg_drscc"]

    fig, axes = plt.subplots(3, 2, figsize=(15, 11))
    axes = axes.flatten()

    plot_one_alignment_subplot(
        axes[0], t_mmw_c, mmw_norm_axis, t_ecg_dtw, ecg_norm_30,
        title=f"DTW | {format_seg(result['start_dtw'], result['end_dtw'])} | mmW={result['mmw_HR_dtw']:.2f}",
        shade_start=result["start_dtw"], shade_end=result["end_dtw"]
    )

    plot_one_alignment_subplot(
        axes[1], t_mmw_c, mmw_norm_axis, t_ecg_drscc, ecg_norm_30,
        title=f"DR-SCC | {format_seg(result['start_drscc'], result['end_drscc'])} | mmW={result['mmw_HR_drscc']:.2f}",
        shade_start=result["start_drscc"], shade_end=result["end_drscc"]
    )

    plot_one_alignment_subplot(
        axes[2], t_mmw_c, mmw_norm_axis, t_ecg_xcorr, ecg_norm_30,
        title=f"XCORR | {format_seg(result['start_xcorr'], result['end_xcorr'])} | mmW={result['mmw_HR_xcorr']:.2f}",
        shade_start=result["start_xcorr"], shade_end=result["end_xcorr"]
    )

    plot_one_alignment_subplot(
        axes[3], t_mmw_c, mmw_norm_axis, t_ecg_gcc, ecg_norm_30,
        title=f"GCC-PHAT | {format_seg(result['start_gccphat'], result['end_gccphat'])} | mmW={result['mmw_HR_gccphat']:.2f}",
        shade_start=result["start_gccphat"], shade_end=result["end_gccphat"]
    )

    plot_one_alignment_subplot(
        axes[4], t_mmw_c, mmw_norm_axis, t_ecg_peaks, ecg_norm_30,
        title=f"Peaks+Drift | {format_seg(result['start_peaks_drift'], result['end_peaks_drift'])} | mmW={result['mmw_HR_peaks_drift']:.2f}",
        shade_start=result["start_peaks_drift"], shade_end=result["end_peaks_drift"]
    )

    axes[5].axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", frameon=False)
    fig.suptitle(f"{cond_name} / {sample_name}", fontsize=14)
    plt.tight_layout()

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [30]:
# =========================================================
# 輸出欄位
# 保留原本：
# sample_name, mmw_HR_raw(or mmw_HR_raw), ECG_HR
# 只額外補 5 種對齊欄位
# =========================================================
OUTPUT_COLUMNS = [
    "sample_name",
    "mmw_HR_raw",
    "ECG_HR",
    blank_col(1),

    "seg_dtw",
    "mmw_HR_dtw",
    blank_col(2),

    "seg_drscc",
    "mmw_HR_drscc",
    blank_col(3),

    "seg_xcorr",
    "mmw_HR_xcorr",
    blank_col(4),

    "seg_gccphat",
    "mmw_HR_gccphat",
    blank_col(5),

    "seg_peaks_drift",
    "mmw_HR_peaks_drift",
]


def get_existing_core_cols(df: pd.DataFrame) -> Dict[str, str]:
    """
    自動辨識原本 csv 的核心欄位名稱
    """
    cols_map = {c.strip().lower(): c for c in df.columns}

    sample_col = cols_map.get("sample_name", None)

    raw_col = None
    for cand in ["mmw_hr_raw", "mmw_hr_raw ", "mmw_hr_raw".lower(), "mmw_hr_raw".upper()]:
        if cand in cols_map:
            raw_col = cols_map[cand]
            break

    # 更保險：直接找所有 lower 後等於 mmw_hr_raw 的欄位
    if raw_col is None:
        for c in df.columns:
            if c.strip().lower() == "mmw_hr_raw":
                raw_col = c
                break

    ecg_col = None
    for c in df.columns:
        if c.strip().lower() == "ecg_hr":
            ecg_col = c
            break

    return {
        "sample_name": sample_col,
        "mmw_HR_raw": raw_col,
        "ECG_HR": ecg_col,
    }


def update_row_with_alignment_info(row: pd.Series, result: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(row)

    # 不動原本的 mmw_HR_raw / ECG_HR
    out[blank_col(1)] = ""
    out["seg_dtw"] = format_seg(result["start_dtw"], result["end_dtw"])
    out["mmw_HR_dtw"] = result["mmw_HR_dtw"]
    out[blank_col(2)] = ""

    out["seg_drscc"] = format_seg(result["start_drscc"], result["end_drscc"])
    out["mmw_HR_drscc"] = result["mmw_HR_drscc"]
    out[blank_col(3)] = ""

    out["seg_xcorr"] = format_seg(result["start_xcorr"], result["end_xcorr"])
    out["mmw_HR_xcorr"] = result["mmw_HR_xcorr"]
    out[blank_col(4)] = ""

    out["seg_gccphat"] = format_seg(result["start_gccphat"], result["end_gccphat"])
    out["mmw_HR_gccphat"] = result["mmw_HR_gccphat"]
    out[blank_col(5)] = ""

    out["seg_peaks_drift"] = format_seg(result["start_peaks_drift"], result["end_peaks_drift"])
    out["mmw_HR_peaks_drift"] = result["mmw_HR_peaks_drift"]

    return out

In [ ]:
# =========================================================
# 單一 condition 批次處理
# 直接讀原本 result csv，更新後覆寫
# =========================================================
def process_condition_alignment(cond_dir: Path, out_csv: Path, save_fig: bool = True) -> pd.DataFrame:
    if not out_csv.exists():
        raise FileNotFoundError(f"找不到原本 result csv：{out_csv}")

    df_base = pd.read_csv(out_csv)

    core_cols = get_existing_core_cols(df_base)

    if core_cols["sample_name"] is None:
        raise ValueError(f"{out_csv} 缺少 sample_name 欄位")
    if core_cols["mmw_HR_raw"] is None:
        raise ValueError(f"{out_csv} 缺少 mmw_HR_raw / mmW_HR_raw 欄位")
    if core_cols["ECG_HR"] is None:
        raise ValueError(f"{out_csv} 缺少 ECG_HR 欄位")

    sample_col = core_cols["sample_name"]
    raw_col = core_cols["mmw_HR_raw"]
    ecg_col = core_cols["ECG_HR"]

    # 統一欄名成你最後要的格式
    rename_map = {}
    if sample_col != "sample_name":
        rename_map[sample_col] = "sample_name"
    if raw_col != "mmw_HR_raw":
        rename_map[raw_col] = "mmw_HR_raw"
    if ecg_col != "ECG_HR":
        rename_map[ecg_col] = "ECG_HR"

    if rename_map:
        df_base = df_base.rename(columns=rename_map)

    # 確保新增欄位存在
    for c in OUTPUT_COLUMNS:
        if c not in df_base.columns:
            df_base[c] = ""

    
    for idx, row in df_base.iterrows(): 
        sample_name = str(row["sample_name"]).strip()

        sample_dir = cond_dir / sample_name
        mmw_csv = find_waveform_csv(sample_dir)
        ecg_csv = find_ecg_csv(cond_dir, sample_name)

        if mmw_csv is None:
            print(f"[WARN] 找不到 waveform.csv: {sample_dir}")
            continue
        if ecg_csv is None:
            print(f"[WARN] 找不到 ECG csv: {cond_dir / 'raw' / 'ECG' / (sample_name + '.csv')}")
            continue

        try:
            result = analyze_one_sample(mmw_csv, ecg_csv)
            updated = update_row_with_alignment_info(row, result)

            for c in OUTPUT_COLUMNS:
                df_base.at[idx, c] = updated.get(c, "")

            if save_fig:
                fig_path = sample_dir / "alignment_5methods.png"
                plot_all_5_methods(
                    result,
                    sample_name=sample_name,
                    cond_name=cond_dir.name,
                    save_path=fig_path,
                )

            print(f"[OK] {cond_dir.name} | {sample_name}")

        except Exception as e:
            print(f"[WARN] {cond_dir.name} | {sample_name} | {e}")


    df_base = df_base[OUTPUT_COLUMNS]
    df_base.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return df_base

In [32]:
# =========================================================
# 全部批次處理
# =========================================================
def batch_process_all_conditions_alignment(
    data_dir: Path,
    cond_names: List[str],
    out_dir: Path,
    save_fig: bool = True,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    for cond in cond_names:
        cond_dir = data_dir / cond
        out_csv = out_dir / f"{cond}.csv"

        if not cond_dir.exists():
            print(f"[SKIP] 找不到資料夾: {cond_dir}")
            continue

        if not out_csv.exists():
            print(f"[SKIP] 找不到原本 csv: {out_csv}")
            continue

        df_out = process_condition_alignment(cond_dir, out_csv, save_fig=save_fig)
        print(f"[DONE] {cond}: {len(df_out)} 筆 -> {out_csv}")

In [33]:
if __name__ == "__main__":
    batch_process_all_conditions_alignment(
    data_dir=DATA_DIR,
    cond_names=COND_NAMES,
    out_dir=OUT_DIR,
    save_fig=True,
)

C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C

[OK] AWR_steady | sample_0
[OK] AWR_steady | sample_1
[OK] AWR_steady | sample_2
[OK] AWR_steady | sample_3
[OK] AWR_steady | sample_4
[OK] AWR_steady | sample_5
[OK] AWR_steady | sample_6
[OK] AWR_steady | sample_7
[OK] AWR_steady | sample_8
[OK] AWR_steady | sample_9
[OK] AWR_steady | sample_10
[OK] AWR_steady | sample_11
[OK] AWR_steady | sample_12
[OK] AWR_steady | sample_13
[OK] AWR_steady | sample_14
[OK] AWR_steady | sample_15
[OK] AWR_steady | sample_16
[OK] AWR_steady | sample_17
[OK] AWR_steady | sample_18
[OK] AWR_steady | sample_19
[OK] AWR_steady | sample_20
[OK] AWR_steady | sample_21
[OK] AWR_steady | sample_22
[OK] AWR_steady | sample_23
[OK] AWR_steady | sample_24
[OK] AWR_steady | sample_25
[OK] AWR_steady | sample_26
[OK] AWR_steady | sample_27
[OK] AWR_steady | sample_28
[OK] AWR_steady | sample_29
[OK] AWR_steady | sample_30
[OK] AWR_steady | sample_31
[OK] AWR_steady | sample_32
[OK] AWR_steady | sample_33
[OK] AWR_steady | sample_34
[OK] AWR_steady | sample_35
[O

C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.00, 30.00]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated

[OK] AWR_unsteady | sample_0
[OK] AWR_unsteady | sample_1
[OK] AWR_unsteady | sample_2
[OK] AWR_unsteady | sample_3
[OK] AWR_unsteady | sample_4
[OK] AWR_unsteady | sample_5
[OK] AWR_unsteady | sample_6
[OK] AWR_unsteady | sample_7
[OK] AWR_unsteady | sample_8
[OK] AWR_unsteady | sample_9
[OK] AWR_unsteady | sample_10
[OK] AWR_unsteady | sample_11
[OK] AWR_unsteady | sample_12
[OK] AWR_unsteady | sample_13
[OK] AWR_unsteady | sample_14
[OK] AWR_unsteady | sample_15
[OK] AWR_unsteady | sample_16
[OK] AWR_unsteady | sample_17
[OK] AWR_unsteady | sample_18
[OK] AWR_unsteady | sample_19
[OK] AWR_unsteady | sample_20
[OK] AWR_unsteady | sample_21
[OK] AWR_unsteady | sample_22
[OK] AWR_unsteady | sample_23
[OK] AWR_unsteady | sample_24
[OK] AWR_unsteady | sample_25
[OK] AWR_unsteady | sample_26
[OK] AWR_unsteady | sample_27
[OK] AWR_unsteady | sample_28
[OK] AWR_unsteady | sample_29
[OK] AWR_unsteady | sample_30
[OK] AWR_unsteady | sample_31
[OK] AWR_unsteady | sample_32
[OK] AWR_unsteady | 

C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[30.00, 60.00]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = update

[OK] IWR_steady | sample_0
[OK] IWR_steady | sample_1
[OK] IWR_steady | sample_2
[OK] IWR_steady | sample_3
[OK] IWR_steady | sample_4
[OK] IWR_steady | sample_5
[OK] IWR_steady | sample_6
[OK] IWR_steady | sample_7
[OK] IWR_steady | sample_8
[OK] IWR_steady | sample_9
[OK] IWR_steady | sample_10
[OK] IWR_steady | sample_11
[OK] IWR_steady | sample_12
[OK] IWR_steady | sample_13
[OK] IWR_steady | sample_14
[OK] IWR_steady | sample_15
[OK] IWR_steady | sample_16
[OK] IWR_steady | sample_17
[OK] IWR_steady | sample_18
[OK] IWR_steady | sample_19
[OK] IWR_steady | sample_20
[OK] IWR_steady | sample_21
[OK] IWR_steady | sample_22
[OK] IWR_steady | sample_23
[OK] IWR_steady | sample_24
[OK] IWR_steady | sample_25
[OK] IWR_steady | sample_26
[OK] IWR_steady | sample_27
[OK] IWR_steady | sample_28
[OK] IWR_steady | sample_29
[OK] IWR_steady | sample_30
[OK] IWR_steady | sample_31
[OK] IWR_steady | sample_32
[OK] IWR_steady | sample_33
[OK] IWR_steady | sample_34
[OK] IWR_steady | sample_35
[O

C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[23.00, 53.00]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = updated.get(c, "")
C:\Users\YShane11\AppData\Local\Temp\ipykernel_23016\1904522302.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_base.at[idx, c] = update

[OK] IWR_unsteady | sample_0
[OK] IWR_unsteady | sample_1
[OK] IWR_unsteady | sample_2
[OK] IWR_unsteady | sample_3
[OK] IWR_unsteady | sample_4
[OK] IWR_unsteady | sample_5
[OK] IWR_unsteady | sample_6
[OK] IWR_unsteady | sample_7
[OK] IWR_unsteady | sample_8
[OK] IWR_unsteady | sample_9
[OK] IWR_unsteady | sample_10
[OK] IWR_unsteady | sample_11
[OK] IWR_unsteady | sample_12
[OK] IWR_unsteady | sample_13
[OK] IWR_unsteady | sample_14
[OK] IWR_unsteady | sample_15
[OK] IWR_unsteady | sample_16
[OK] IWR_unsteady | sample_17
[OK] IWR_unsteady | sample_18
[OK] IWR_unsteady | sample_19
[OK] IWR_unsteady | sample_20
[OK] IWR_unsteady | sample_21
[OK] IWR_unsteady | sample_22
[OK] IWR_unsteady | sample_23
[OK] IWR_unsteady | sample_24
[OK] IWR_unsteady | sample_25
[OK] IWR_unsteady | sample_26
[OK] IWR_unsteady | sample_27
[OK] IWR_unsteady | sample_28
[OK] IWR_unsteady | sample_29
[OK] IWR_unsteady | sample_30
[OK] IWR_unsteady | sample_31
[OK] IWR_unsteady | sample_32
[OK] IWR_unsteady | 

DTW (Dynamic Time Warping)
允許訊號在時間軸上局部伸縮，尋找 mmWave 與 ECG 之間整體形狀最相似的對應區段。

DR-SCC (Drift-Regularized Sliding Cross-Correlation)
將訊號分成多個滑動視窗逐段做相關比對，並限制相鄰視窗的位移變化，以避免不合理的時間跳動。

XCORR (Cross-Correlation)
計算兩段訊號的整體互相關，找出使相似度最大的固定時間延遲。

GCC-PHAT (Generalized Cross-Correlation with Phase Transform)
在互相關前對頻譜做相位正規化，減少振幅差異影響，使延遲估計更穩定。

Peaks + Drift
先利用峰值對齊找出大致時間位置，再用線性漂移模型修正隨時間累積的對齊偏差。